# Linear And Kernelized Attention

This notebook builds a small linear-attention reference path, proves the associative regrouping that removes the `T x T` score matrix for positive kernels, and measures how a Performer-style positive random-feature approximation tracks exact softmax attention as feature dimension grows.

## Motivation

Exact attention forms a query-key score for every allowed pair of positions. That is the right reference mechanism, but it becomes expensive when sequence length grows because the score tensor has quadratic size in `T`. Linear attention asks whether we can replace the pairwise score matrix with a positive feature map `phi` such that `k(q, k') = phi(q)^T phi(k')`, then regroup the sums so sequence cost becomes linear in `T` for fixed feature dimension. The core research question here is two-part: when is the regrouping algebraically exact for a chosen kernel, and how well do positive random features approximate the exact softmax kernel in practice [@attention_is_all_you_need; @performer_2020]?

## Hypothesis

If the feature map is positive, then kernelized attention can be computed from prefix summaries without materializing a full causal `T x T` matrix. For softmax-style kernels, a positive random-feature approximation should approach exact attention as the feature dimension increases, although the approximation can still be noisy on small feature budgets.

## Baseline

We use Task 4 exact causal softmax attention as the empirical baseline. Before comparing against that baseline, we also compare the deterministic `elu(x) + 1` feature map against an explicit kernelized reference built from the full feature Gram matrix. That isolates implementation correctness from approximation quality.

## Metric

The main metric is output error versus exact causal softmax attention, reported as mean absolute error and maximum absolute error over all output entries. For the linear reference path itself, the correctness metric is equality with the explicit kernelized reference, and the structural metric is whether the trace records any intermediate with trailing shape `(T, T)`.

## Mathematical derivation

Let `phi : R^d -> R_+^m` be a positive feature map and define the kernel `k(q_i, k_j) = phi(q_i)^T phi(k_j)`. Ignoring normalization for a moment, the value update for query `i` is

$$\sum_{j=1}^T k(q_i, k_j) v_j = \sum_{j=1}^T \phi(q_i)^T \phi(k_j) v_j = \phi(q_i)^T \left(\sum_{j=1}^T \phi(k_j) v_j^T\right).$$

The expensive pairwise interaction disappears because `phi(q_i)` can be factored outside the sum. After row normalization, noncausal kernel attention becomes

$$y_i = \frac{\phi(q_i)^T \left(\sum_j \phi(k_j) v_j^T\right)}{\phi(q_i)^T \left(\sum_j \phi(k_j)\right)}.$$

For causal attention, queries may only see their prefix, so define running statistics

$$S_i = \sum_{j \le i} \phi(k_j) v_j^T, \qquad z_i = \sum_{j \le i} \phi(k_j).$$

Then each output is

$$y_i = \frac{\phi(q_i)^T S_i}{\phi(q_i)^T z_i},$$

which can be implemented with prefix sums over `phi(k_i) v_i^T` and `phi(k_i)` instead of a full attention matrix. This is exact for the chosen kernelized model. To approximate exact softmax attention, Performer-style positive random features target the exponential kernel `exp(q_i^T k_j / sqrt(d))` so the same associative structure applies approximately [@performer_2020].

In [ ]:
from pathlib import Path
import sys

import torch

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
SRC_PATH = ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

from llm_from_scratch.research.attention_math import exact_attention
from llm_from_scratch.research.linear_attention import (
    LinearAttentionTrace,
    causal_linear_attention,
    elu_plus_one_feature_map,
    kernel_attention_via_feature_map,
    softmax_random_feature_attention,
)

torch.manual_seed(0)
torch.set_printoptions(precision=5, sci_mode=False)


## PyTorch implementation

The module exposes three useful paths: an explicit kernelized reference that does materialize pairwise feature scores, a causal linear reference that uses prefix summaries and a trace object, and a positive random-feature approximation for exact softmax attention. The testable contract is that the causal linear path records its intermediates and never reports a `(T, T)` tensor.

In [ ]:
def make_causal_mask(length: int, device: torch.device | None = None) -> torch.Tensor:
    positions = torch.arange(length, device=device)
    return positions[:, None] >= positions[None, :]

q = 0.35 * torch.randn(1, 6, 4, dtype=torch.float64)
k = 0.35 * torch.randn(1, 6, 4, dtype=torch.float64)
v = torch.randn(1, 6, 3, dtype=torch.float64)
mask = make_causal_mask(q.shape[-2], device=q.device)

trace = LinearAttentionTrace()
kernel_reference = kernel_attention_via_feature_map(
    q,
    k,
    v,
    feature_map=elu_plus_one_feature_map,
    causal=True,
)
linear_reference = causal_linear_attention(
    q,
    k,
    v,
    feature_map=elu_plus_one_feature_map,
    trace=trace,
)
exact_output, exact_weights = exact_attention(q, k, v, mask=mask)

trace.intermediate_shapes


## Numerical checks

Before interpreting approximation error, verify two facts: the associative linear path exactly matches the explicit kernelized reference for the deterministic positive feature map, and the recorded intermediates do not include a sequence-square matrix. Then compare the random-feature approximation against exact softmax attention.

In [ ]:
assert torch.allclose(linear_reference, kernel_reference, atol=1e-10, rtol=1e-10)
assert trace.sequence_square_matrices == []
assert any(name == 'feature_kv_prefix' for name, _ in trace.intermediate_shapes)
assert exact_weights.shape[-2:] == (q.shape[-2], k.shape[-2])

feature_dims = [8, 16, 32, 64, 128, 256]
error_summary = []
for num_features in feature_dims:
    approx = softmax_random_feature_attention(
        q,
        k,
        v,
        num_features=num_features,
        seed=0,
        causal=True,
    )
    error_summary.append({
        'num_features': num_features,
        'mean_abs_error': round((approx - exact_output).abs().mean().item(), 6),
        'max_abs_error': round((approx - exact_output).abs().max().item(), 6),
    })

assert error_summary[-1]['mean_abs_error'] < error_summary[0]['mean_abs_error']
error_summary


## Ablations

The only ablation here is feature dimension. That isolates the approximation-variance story: the algorithmic structure stays the same, the sequence cost remains linear in `T` for fixed feature count, and only the number of random features changes. If error does not improve with larger feature dimension on a bounded toy setup, the implementation or the scaling is suspect.

In [ ]:
best_case = min(error_summary, key=lambda row: row['mean_abs_error'])
worst_case = max(error_summary, key=lambda row: row['mean_abs_error'])
{
    'worst_feature_budget': worst_case,
    'best_feature_budget': best_case,
    'linear_trace_square_matrices': trace.sequence_square_matrices,
}


## Assumptions

This notebook assumes small-norm queries and keys so the positive random-feature exponentials stay numerically well behaved without extra stabilization tricks. It also assumes that output error on a deterministic toy example is a useful proxy for approximation quality, which is weaker than a task-level accuracy study.

## Risks

A low mean output error can still hide a bad failure on a rare token or head. The deterministic `elu(x) + 1` feature map is also not an approximation of exact softmax; it is a different kernelized model whose value here is to validate the associative implementation path. Finally, small synthetic examples can understate variance that appears on larger-norm or higher-dimensional inputs.

## Failure criteria

Treat the experiment as failed if any of the following happens: the causal linear path disagrees with the explicit kernelized reference, the trace reports a `(T, T)` intermediate for the linear path, or the larger random-feature budgets do not improve over the smallest one on this bounded setup. Those outcomes would mean we have not yet separated algebraic correctness from approximation quality.

## Exercises

1. Re-derive the causal prefix update and state the shapes of both running statistics.
2. Explain why the deterministic `elu(x) + 1` path can be algebraically exact for its kernel while still failing to match exact softmax attention.
3. Increase or decrease the random-feature dimension sweep and record how mean and max error move for the same seed.

Work the companion exercise sheet after running the notebook:

- `exercises/research/16_linear_kernel_attention.md`
- `exercises/research/solutions/16_linear_kernel_attention_solutions.md`


## References

- Vaswani et al., *Attention Is All You Need* [@attention_is_all_you_need].
- Choromanski et al., *Rethinking Attention with Performers* [@performer_2020].